In [ ]:
from __future__ import annotations

import gc
import math
import os
import random
from collections import Counter
from dataclasses import dataclass
from typing import Any, Dict, List, Literal, Optional, Tuple

import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer

SEED = 42
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
from from_n3c import *

# Keep the complete three-character ICD-10 category vocabulary.
EXPECTED_NUM_CONCEPTS = 2347
df_concepts = pd.read_csv('../AdaptivePooling_MLHC/df_icd10.csv')
df_concepts['code'] = (
    df_concepts['code'].astype(str).str.upper()
    .str.replace(r'[^A-Z0-9]', '', regex=True)
)
df_concepts = (
    df_concepts[df_concepts['code'].str.fullmatch(r'[A-Z][0-9][A-Z0-9]')]
    .drop_duplicates('code')
    .reset_index(drop=True)
)

concepts = [
    {
        'id': row.code,
        'text': f"{row.code}: {row['name']}",
        'group': str(row.idx_section + 1),
    }
    for _, row in df_concepts.iterrows()
]
print(f"Loaded {len(concepts):,} unique three-character ICD-10 concepts.")
if len(concepts) != EXPECTED_NUM_CONCEPTS:
    print(
        f"WARNING: the manuscript used {EXPECTED_NUM_CONCEPTS:,} concepts, but this CSV produced "
        f"{len(concepts):,}. Check that df_icd10.csv is the same vocabulary release."
    )

with open("../AdaptivePooling_MLHC/ds_train_chest_trauma_ner.json", "r") as json_file:
    train_samples = json.load(json_file)
with open("../AdaptivePooling_MLHC/ds_dev_chest_trauma_ner.json", "r") as json_file:
    dev_samples = json.load(json_file)
with open("../AdaptivePooling_MLHC/ds_test_chest_trauma_ner.json", "r") as json_file:
    val_samples = json.load(json_file)

for samples in (train_samples, dev_samples, val_samples):
    for sample in samples:
        sample['label'] = 0 if sample['label'] < 3 else 1


In [ ]:
train_samples[0]

### Revision: note-level supervision over the full ICD-10 category vocabulary

This version keeps all unique three-character ICD-10 categories from `df_icd10.csv` (2,347 in the manuscript) and makes only the grounding changes needed for the revision:

1. Each sample's `concepts` list is mapped to a multi-hot note-level target over the complete vocabulary.
2. Concept-wise sparsemax/softmax, the NULL gate, and top-k concept pruning are replaced by independent sigmoid token–concept scores.
3. Token logits are pooled with a length-normalized smooth maximum.
4. Outcome loss is jointly optimized with a balanced note-level concept-presence loss.
5. Concise concept descriptions (`CODE: name`) are used. During four simultaneous warm-up epochs, the shared encoder is optimized by the original `[CLS]` outcome loss and the note-level concept loss in the same minibatch; concept embeddings are refreshed at the start of each epoch.

**Label assumption:** `sample["concepts"]` exhaustively lists affirmed, patient-specific concepts present in the note. Therefore, unlisted vocabulary concepts are treated as absent. Full codes such as `S22.3` fall back to their three-character category `S22`.

6. The post-warm-up outcome stage grid-searches equal, reduced, frozen, and freeze-then-unfreeze learning schedules for the grounding parameters. Development AUROC, concept micro-F1, and pure presence hit@1 are tracked after every epoch.


In [ ]:
# ============================================================
# Dataset and note-level concept labels
# ============================================================
def normalize_icd10_code(code: Any) -> str:
    """Uppercase alphanumeric ICD-10 code without punctuation/prefix."""
    if code is None:
        return ""
    value = "".join(ch for ch in str(code).upper().strip() if ch.isalnum())
    for prefix in ("ICD10CM", "ICD10"):
        if value.startswith(prefix):
            value = value[len(prefix):]
            break
    return value


def build_concept_code_index(vocab: List[Dict[str, Any]]) -> Dict[str, int]:
    """Map exact codes and unambiguous three-character categories to row indices."""
    exact: Dict[str, int] = {}
    category_to_indices: Dict[str, set[int]] = {}

    for i, concept in enumerate(vocab):
        code = normalize_icd10_code(concept["id"])
        if not code:
            raise ValueError(f"Empty concept code at vocabulary index {i}.")
        if code in exact and exact[code] != i:
            raise ValueError(f"Duplicate concept code after normalization: {code}")
        exact[code] = i
        category_to_indices.setdefault(code[:3], set()).add(i)

    mapping = dict(exact)
    for category, indices in category_to_indices.items():
        if len(indices) == 1:
            mapping.setdefault(category, next(iter(indices)))
    return mapping


def _concept_code_from_annotation(annotation: Any) -> Any:
    if isinstance(annotation, dict):
        for key in ("code", "id", "concept_id", "icd10", "ICD10"):
            if annotation.get(key) is not None:
                return annotation[key]
        return None
    if isinstance(annotation, (list, tuple)):
        if len(annotation) >= 2:
            return annotation[1]
        return annotation[0] if annotation else None
    return annotation


def extract_concept_indices(
    annotations: List[Any],
    concept_to_idx: Dict[str, int],
) -> Tuple[List[int], List[str]]:
    indices, unmatched = set(), []
    for annotation in annotations or []:
        code = normalize_icd10_code(_concept_code_from_annotation(annotation))
        idx = concept_to_idx.get(code)
        if idx is None and len(code) >= 3:
            idx = concept_to_idx.get(code[:3])
        if idx is None:
            if code:
                unmatched.append(code)
        else:
            indices.add(idx)
    return sorted(indices), unmatched


def concept_label_statistics(
    samples: List[Dict[str, Any]],
    concept_to_idx: Dict[str, int],
    num_concepts: int,
) -> Tuple[torch.Tensor, Counter]:
    counts = torch.zeros(num_concepts, dtype=torch.float32)
    unmatched = Counter()
    for sample in samples:
        indices, missing = extract_concept_indices(sample.get("concepts", []), concept_to_idx)
        if indices:
            counts[indices] += 1.0
        unmatched.update(missing)
    return counts, unmatched


class TextConceptDataset(Dataset):
    def __init__(
        self,
        samples: List[Dict[str, Any]],
        concept_to_idx: Dict[str, int],
    ):
        self.samples = samples
        self.concept_indices: List[List[int]] = []

        for i, sample in enumerate(samples):
            missing = {"txt", "label", "concepts"} - set(sample)
            if missing:
                raise KeyError(f"Sample {i} is missing keys: {sorted(missing)}")
            indices, _ = extract_concept_indices(sample["concepts"], concept_to_idx)
            self.concept_indices.append(indices)

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        sample = self.samples[idx]
        return {
            "txt": sample["txt"],
            "label": sample["label"],
            "concept_indices": self.concept_indices[idx],
        }


def _infer_label_tensor(labels: List[Any]) -> torch.Tensor:
    first = labels[0]
    if isinstance(first, torch.Tensor):
        first = first.detach().cpu().tolist() if first.ndim else first.item()
    if isinstance(first, np.ndarray):
        first = first.tolist()

    if isinstance(first, (list, tuple)):
        return torch.tensor([
            x.detach().cpu().tolist() if isinstance(x, torch.Tensor)
            else x.tolist() if isinstance(x, np.ndarray)
            else x
            for x in labels
        ], dtype=torch.float32)
    if isinstance(first, (bool, np.bool_, int, np.integer)):
        return torch.tensor(labels, dtype=torch.long)
    return torch.tensor(labels, dtype=torch.float32)


def make_collate_fn(
    tokenizer,
    num_concepts: int,
    max_length: int = 512,
    return_text: bool = False,
):
    def collate(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
        texts = [item["txt"] for item in batch]
        encoded = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )

        concept_labels = torch.zeros(len(batch), num_concepts, dtype=torch.float32)
        for i, item in enumerate(batch):
            if item["concept_indices"]:
                concept_labels[i, item["concept_indices"]] = 1.0

        output = {
            **encoded,
            "labels": _infer_label_tensor([item["label"] for item in batch]),
            "concept_labels": concept_labels,
        }
        if return_text:
            output["txt"] = texts
        return output

    return collate


def make_dataloader(
    samples: List[Dict[str, Any]],
    tokenizer,
    concept_to_idx: Dict[str, int],
    num_concepts: int,
    batch_size: int,
    shuffle: bool,
    max_length: int = 512,
    num_workers: int = 2,
) -> DataLoader:
    return DataLoader(
        TextConceptDataset(samples, concept_to_idx),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        collate_fn=make_collate_fn(tokenizer, num_concepts, max_length),
    )


# ============================================================
# Concept vocabulary and embeddings
# ============================================================
def concepts_to_texts_and_groups(
    vocab: List[Dict[str, Any]],
    text_key: str = "text",
    group_key: str = "group",
) -> Tuple[List[str], List[List[int]], List[str]]:
    concept_texts = [str(concept[text_key]) for concept in vocab]
    group_to_indices: Dict[str, List[int]] = {}
    for i, concept in enumerate(vocab):
        group_to_indices.setdefault(str(concept[group_key]), []).append(i)
    group_names = list(group_to_indices)
    return concept_texts, [group_to_indices[name] for name in group_names], group_names


@torch.no_grad()
def build_concept_embeddings(
    concept_texts: List[str],
    tokenizer,
    text_encoder: nn.Module,
    device: torch.device,
    batch_size: int = 4,
    max_length: int = 256,
    pooling: Literal["cls", "mean"] = "cls",
) -> torch.Tensor:
    text_encoder.eval()
    embeddings = []

    for start in range(0, len(concept_texts), batch_size):
        tokens = tokenizer(
            concept_texts[start:start + batch_size],
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(device)
        hidden = text_encoder(**tokens).last_hidden_state
        if pooling == "cls":
            pooled = hidden[:, 0]
        else:
            mask = tokens["attention_mask"].unsqueeze(-1)
            pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1)
        embeddings.append(pooled.cpu())

    return torch.cat(embeddings, dim=0)


# ============================================================
# Independent token-concept matching + smooth note-level pooling
# ============================================================
@dataclass
class AVOOutput:
    logits: torch.Tensor                 # outcome logits, (B, K)
    token_logits: torch.Tensor           # independent token-concept logits, (B, L, C)
    concept_logits: torch.Tensor         # note-level concept logits, (B, C)
    A: torch.Tensor                      # token-level concept probabilities, (B, L, C)
    A_pool: torch.Tensor                 # note-level concept probabilities, (B, C)
    V: torch.Tensor                      # concept values, (C, dv)
    O: torch.Tensor                      # outcome projection, (dv, K)
    AV_pool: torch.Tensor                # pooled representation, (B, dv)
    sim: torch.Tensor                    # raw projected cosine similarities, (B, L, C)


class MentionAlignedAVOHead(nn.Module):
    """
    Each token-concept pair is an independent Bernoulli-style match.
    Note-level concept presence is a length-normalized smooth maximum.
    The outcome remains decomposable as A_pool @ (V @ O) + bias.
    """
    def __init__(
        self,
        concept_emb: torch.Tensor,
        dv: int,
        num_outputs: int,
        temperature: float = 0.07,
        pool_temperature: float = 0.10,
        concept_bias_init: Optional[torch.Tensor] = None,
        freeze_concepts: bool = True,
        use_bias: bool = True,
    ):
        super().__init__()
        if concept_emb.ndim != 2:
            raise ValueError("concept_emb must have shape (C, H).")
        if temperature <= 0 or pool_temperature <= 0:
            raise ValueError("Temperatures must be positive.")

        self.C, self.H = concept_emb.shape
        self.dv = dv
        self.temperature = float(temperature)
        self.pool_temperature = float(pool_temperature)

        if freeze_concepts:
            self.register_buffer("concept_emb", concept_emb.detach().clone())
        else:
            self.concept_emb = nn.Parameter(concept_emb.detach().clone())

        self.Wq = nn.Linear(self.H, self.H, bias=False)
        self.Wk = nn.Linear(self.H, self.H, bias=False)
        nn.init.eye_(self.Wq.weight)
        nn.init.eye_(self.Wk.weight)

        if concept_bias_init is None:
            concept_bias_init = torch.zeros(self.C)
        if tuple(concept_bias_init.shape) != (self.C,):
            raise ValueError(f"concept_bias_init must have shape ({self.C},).")
        self.concept_bias = nn.Parameter(concept_bias_init.detach().clone().float())

        self.Wv = nn.Linear(self.H, dv, bias=False)
        self.O = nn.Parameter(torch.randn(dv, num_outputs) * 0.02)
        self.bias = nn.Parameter(torch.zeros(num_outputs)) if use_bias else None

    def forward(
        self,
        token_embs: torch.Tensor,
        token_mask: Optional[torch.Tensor] = None,
    ) -> AVOOutput:
        batch_size, seq_len, hidden_size = token_embs.shape
        if hidden_size != self.H:
            raise ValueError(f"Token dimension {hidden_size} != concept dimension {self.H}.")

        if token_mask is None:
            token_mask = torch.ones(batch_size, seq_len, dtype=torch.bool, device=token_embs.device)
        else:
            token_mask = token_mask.bool()

        # Ensure smooth pooling has at least one valid position for an empty note.
        empty = token_mask.sum(dim=1) == 0
        if empty.any():
            token_mask = token_mask.clone()
            token_mask[empty, 0] = True

        q = F.normalize(self.Wq(token_embs), p=2, dim=-1)
        k = F.normalize(self.Wk(self.concept_emb), p=2, dim=-1)
        sim = torch.einsum("blh,ch->blc", q, k)

        token_logits = sim / self.temperature + self.concept_bias.view(1, 1, -1)
        pooled_source = token_logits.masked_fill(~token_mask.unsqueeze(-1), float("-inf"))

        valid_length = token_mask.sum(dim=1, keepdim=True).to(token_logits.dtype)
        concept_logits = self.pool_temperature * (
            torch.logsumexp(pooled_source / self.pool_temperature, dim=1)
            - valid_length.log()
        )
        concept_presence = torch.sigmoid(concept_logits)

        token_presence = torch.sigmoid(token_logits)
        token_presence = token_presence.masked_fill(~token_mask.unsqueeze(-1), 0.0)

        V = self.Wv(self.concept_emb)
        AV_pool = concept_presence @ V
        outcome_logits = AV_pool @ self.O
        if self.bias is not None:
            outcome_logits = outcome_logits + self.bias

        return AVOOutput(
            logits=outcome_logits,
            token_logits=pooled_source,
            concept_logits=concept_logits,
            A=token_presence,
            A_pool=concept_presence,
            V=V,
            O=self.O,
            AV_pool=AV_pool,
            sim=sim,
        )


class MentionAlignedAVOModel(nn.Module):
    def __init__(
        self,
        text_encoder: nn.Module,
        head: MentionAlignedAVOHead,
        special_token_ids: Optional[List[int]] = None,
        freeze_text_encoder: bool = True,
    ):
        super().__init__()
        self.text_encoder = text_encoder
        self.head = head
        self.special_token_ids = special_token_ids or []
        self.freeze_text_encoder = freeze_text_encoder

        if freeze_text_encoder:
            for parameter in self.text_encoder.parameters():
                parameter.requires_grad = False
            self.text_encoder.eval()

    def train(self, mode: bool = True):
        super().train(mode)
        if self.freeze_text_encoder:
            self.text_encoder.eval()
        return self

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: Optional[torch.Tensor] = None,
    ) -> Tuple[AVOOutput, torch.Tensor]:
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids

        if self.freeze_text_encoder:
            with torch.no_grad():
                token_embs = self.text_encoder(**kwargs).last_hidden_state
        else:
            token_embs = self.text_encoder(**kwargs).last_hidden_state

        token_mask = attention_mask.bool()
        for token_id in self.special_token_ids:
            token_mask &= input_ids != token_id

        return self.head(token_embs, token_mask), token_mask


# ============================================================
# Existing outcome regularization, retained with no NULL row
# ============================================================
@dataclass
class GroupLassoConfig:
    lambda_group_lasso: float = 1e-3
    use_sqrt_group_size_weight: bool = True
    eps: float = 1e-8


def prepare_group_index_tensors(
    groups: List[List[int]],
    device: torch.device,
) -> List[torch.Tensor]:
    return [
        torch.tensor(group, dtype=torch.long, device=device)
        for group in groups if group
    ]


def group_lasso_penalty_on_beta(
    beta: torch.Tensor,
    group_row_indices: List[torch.Tensor],
    cfg: GroupLassoConfig,
) -> torch.Tensor:
    if cfg.lambda_group_lasso <= 0:
        return beta.new_zeros(())

    penalty = beta.new_zeros(())
    for indices in group_row_indices:
        beta_group = beta.index_select(0, indices)
        group_norm = torch.sqrt((beta_group ** 2).sum() + cfg.eps)
        weight = math.sqrt(indices.numel()) if cfg.use_sqrt_group_size_weight else 1.0
        penalty = penalty + weight * group_norm
    return cfg.lambda_group_lasso * penalty


# ============================================================
# Task losses and evaluation
# ============================================================
def infer_task_and_outputs(samples: List[Dict[str, Any]]) -> Tuple[str, int]:
    label = samples[0]["label"]
    if isinstance(label, torch.Tensor) and label.ndim > 0:
        return "multilabel", int(label.numel())
    if isinstance(label, np.ndarray):
        return "multilabel", int(label.size)
    if isinstance(label, (list, tuple)):
        return "multilabel", len(label)
    if isinstance(label, (float, np.floating)):
        return "regression", 1
    return "multiclass", max(int(sample["label"]) for sample in samples) + 1


def get_loss_fn(task: str):
    if task == "multiclass":
        return nn.CrossEntropyLoss()
    if task == "multilabel":
        return nn.BCEWithLogitsLoss()
    if task == "regression":
        return nn.MSELoss()
    raise ValueError(task)


def sampled_balanced_concept_bce_with_logits(
    logits: torch.Tensor,
    targets: torch.Tensor,
    n_hard_negatives: int = 96,
    n_random_negatives: int = 96,
) -> torch.Tensor:
    """Use every positive plus hard and random negatives from the full vocabulary."""
    positive = targets.bool()
    negative = ~positive
    positive_loss = (
        F.softplus(-logits[positive]).mean() if positive.any() else logits.new_zeros(())
    )

    k_hard = min(n_hard_negatives, logits.shape[1])
    hard_idx = logits.detach().masked_fill(~negative, float("-inf")).topk(k_hard, dim=1).indices
    hard_mask = torch.zeros_like(positive)
    hard_mask.scatter_(1, hard_idx, True)

    k_random = min(n_random_negatives, logits.shape[1])
    random_scores = torch.rand_like(logits).masked_fill(~negative, -1.0)
    random_idx = random_scores.topk(k_random, dim=1).indices
    random_mask = torch.zeros_like(positive)
    random_mask.scatter_(1, random_idx, True)

    selected_negative = negative & (hard_mask | random_mask)
    negative_loss = F.softplus(logits[selected_negative]).mean()
    return 0.5 * (positive_loss + negative_loss)


def _safe_metric(fn, *args, **kwargs) -> float:
    try:
        return float(fn(*args, **kwargs))
    except ValueError:
        return float("nan")


@torch.no_grad()
def evaluate_val(
    model: nn.Module,
    loader: DataLoader,
    task: str,
    device: torch.device,
) -> Dict[str, float]:
    """Outcome metrics plus note-level concept F1 and presence hit@1."""
    model.eval()
    all_probs, all_labels = [], []
    concept_tp = concept_fp = concept_fn = 0
    presence_hit1_correct = presence_hit1_total = 0
    positive_score_sum = negative_score_sum = 0.0
    n_positive = n_negative = 0

    for batch in loader:
        tensors = {
            key: value.to(device)
            for key, value in batch.items()
            if isinstance(value, torch.Tensor)
        }
        out, _ = model(
            input_ids=tensors["input_ids"],
            attention_mask=tensors["attention_mask"],
            token_type_ids=tensors.get("token_type_ids"),
        )

        labels = tensors["labels"]
        if task == "multiclass":
            probs = torch.softmax(out.logits, dim=-1)
        elif task == "multilabel":
            probs = torch.sigmoid(out.logits)
        else:
            probs = out.logits.squeeze(-1)

        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

        targets = tensors["concept_labels"].bool()
        scores = out.A_pool
        predicted = scores >= 0.5

        concept_tp += int((predicted & targets).sum())
        concept_fp += int((predicted & ~targets).sum())
        concept_fn += int((~predicted & targets).sum())

        # Pure concept-presence hit@1, independent of beta/outcome contribution.
        valid_notes = targets.any(dim=1)
        if valid_notes.any():
            top1 = scores.argmax(dim=1, keepdim=True)
            top1_hit = targets.gather(1, top1).squeeze(1)
            presence_hit1_correct += int(top1_hit[valid_notes].sum())
            presence_hit1_total += int(valid_notes.sum())

        positive_score_sum += float(scores[targets].sum())
        negative_score_sum += float(scores[~targets].sum())
        n_positive += int(targets.sum())
        n_negative += int((~targets).sum())

    y_true = np.concatenate(all_labels)
    y_prob = np.concatenate(all_probs)
    metrics: Dict[str, float] = {}

    if task == "multiclass":
        metrics["accuracy"] = float((y_prob.argmax(1) == y_true).mean())
        if y_prob.shape[1] == 2:
            metrics["AUROC"] = _safe_metric(roc_auc_score, y_true, y_prob[:, 1])
            metrics["AUPR"] = _safe_metric(
                average_precision_score, y_true, y_prob[:, 1]
            )
        else:
            metrics["AUROC_macro_ovr"] = _safe_metric(
                roc_auc_score,
                y_true,
                y_prob,
                multi_class="ovr",
                average="macro",
            )
    elif task == "multilabel":
        metrics["AUROC_macro"] = _safe_metric(
            roc_auc_score, y_true, y_prob, average="macro"
        )
        metrics["AUPR_macro"] = _safe_metric(
            average_precision_score, y_true, y_prob, average="macro"
        )

    precision = concept_tp / max(concept_tp + concept_fp, 1)
    recall = concept_tp / max(concept_tp + concept_fn, 1)
    metrics["concept_micro_precision@0.5"] = precision
    metrics["concept_micro_recall@0.5"] = recall
    metrics["concept_micro_F1@0.5"] = (
        2 * precision * recall / max(precision + recall, 1e-12)
    )
    metrics["concept_presence_hit@1"] = (
        presence_hit1_correct / max(presence_hit1_total, 1)
    )
    metrics["concept_presence_hit@1_n"] = float(presence_hit1_total)
    metrics["concept_mean_probability_positive"] = (
        positive_score_sum / max(n_positive, 1)
    )
    metrics["concept_mean_probability_negative"] = (
        negative_score_sum / max(n_negative, 1)
    )
    return metrics


In [ ]:
# Main settings
model_name = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
batch_size = 4
max_length = 512
dv = 256

freeze_text_encoder = True
freeze_concepts = True
attention_temperature = 0.07
pool_temperature = 0.10

lr = 1e-5
joint_warmup_epochs = 4
num_epochs = 2
lambda_concept = 0.5
group_lasso_cfg = GroupLassoConfig(lambda_group_lasso=1e-3)

n_hard_concept_negatives = 96
n_random_concept_negatives = 96

# Grid search for preserving the warmed concept matcher during outcome training.
# Every configuration starts from the same jointly warmed head.
grounding_grid = [
    {
        "name": "same_lr",
        "grounding_lr": lr,
        "outcome_lr": lr,
        "freeze_grounding_epochs": 0,
        "lambda_concept": lambda_concept,
    },
    {
        "name": "grounding_lr_0.1x",
        "grounding_lr": lr * 0.1,
        "outcome_lr": lr,
        "freeze_grounding_epochs": 0,
        "lambda_concept": lambda_concept,
    },
    {
        "name": "grounding_lr_0.01x",
        "grounding_lr": lr * 0.01,
        "outcome_lr": lr,
        "freeze_grounding_epochs": 0,
        "lambda_concept": lambda_concept,
    },
    {
        "name": "freeze_grounding",
        "grounding_lr": 0.0,
        "outcome_lr": lr,
        "freeze_grounding_epochs": num_epochs,
        "lambda_concept": 0.0,
    },
    {
        "name": "freeze_1_then_0.1x",
        "grounding_lr": lr * 0.1,
        "outcome_lr": lr,
        "freeze_grounding_epochs": 1,
        "lambda_concept": lambda_concept,
    },
]

# Select the most grounded checkpoint whose dev AUROC is within this
# tolerance of the best dev AUROC found anywhere in the grid.
outcome_auroc_tolerance = 0.01


In [ ]:
# Vocabulary lookup and concept-label diagnostics
concept_to_idx = build_concept_code_index(concepts)
num_concepts = len(concepts)

concept_counts, unmatched_train_codes = concept_label_statistics(
    train_samples, concept_to_idx, num_concepts
)
n_train = len(train_samples)

# Concept-specific prevalence logits provide a stable starting calibration.
concept_prevalence = (concept_counts + 0.5) / (n_train + 1.0)
concept_bias_init = torch.logit(concept_prevalence.clamp(1e-5, 1 - 1e-5))

for split_name, samples in [
    ("train", train_samples), ("dev", dev_samples), ("val", val_samples)
]:
    split_counts, unmatched_codes = concept_label_statistics(
        samples, concept_to_idx, num_concepts
    )
    print(f"{split_name} label counts:", Counter(sample["label"] for sample in samples))
    print(
        f"{split_name} matched concept labels: {int(split_counts.sum())}; "
        f"unmatched: {sum(unmatched_codes.values())}"
    )
    if unmatched_codes:
        print("  most common unmatched codes:", unmatched_codes.most_common(5))

print(f"Concepts with >=1 train positive: {(concept_counts > 0).sum().item()} / {num_concepts}")
score_tensor_mb = batch_size * max_length * num_concepts * 4 / 1024**2
print(f"Approximate token-concept score tensor per batch (float32): {score_tensor_mb:.1f} MB")


In [ ]:
def set_global_seed(seed: int = 42, deterministic: bool = True) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True)

set_global_seed(SEED, deterministic=True)

In [ ]:
class BlackBoxLM(nn.Module):
    def __init__(self, encoder: nn.Module, num_outputs: int):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(encoder.config.hidden_size, num_outputs)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        hidden = self.encoder(**kwargs).last_hidden_state
        return self.head(hidden[:, 0])


@torch.no_grad()
def evaluate_warmup_concept_f1(
    encoder: nn.Module,
    head: MentionAlignedAVOHead,
    loader: DataLoader,
    special_token_ids: List[int],
    device: torch.device,
    threshold: float = 0.5,
) -> float:
    """Streaming development-set micro-F1 for note-level concept presence."""
    encoder.eval()
    head.eval()
    true_positive = false_positive = false_negative = 0

    for batch in loader:
        tensors = {
            key: value.to(device)
            for key, value in batch.items()
            if isinstance(value, torch.Tensor)
        }
        encoder_kwargs = {
            "input_ids": tensors["input_ids"],
            "attention_mask": tensors["attention_mask"],
        }
        if tensors.get("token_type_ids") is not None:
            encoder_kwargs["token_type_ids"] = tensors["token_type_ids"]

        token_embs = encoder(**encoder_kwargs).last_hidden_state
        token_mask = tensors["attention_mask"].bool()
        for token_id in special_token_ids:
            token_mask &= tensors["input_ids"] != token_id

        predicted = head(token_embs, token_mask).A_pool >= threshold
        targets = tensors["concept_labels"].bool()
        true_positive += int((predicted & targets).sum())
        false_positive += int((predicted & ~targets).sum())
        false_negative += int((~predicted & targets).sum())

    precision = true_positive / max(true_positive + false_positive, 1)
    recall = true_positive / max(true_positive + false_negative, 1)
    return 2 * precision * recall / max(precision + recall, 1e-12)


tokenizer = AutoTokenizer.from_pretrained(model_name)
encoder = AutoModel.from_pretrained(model_name).to(DEVICE)

train_loader = make_dataloader(
    train_samples, tokenizer, concept_to_idx, num_concepts,
    batch_size=batch_size, shuffle=True, max_length=max_length
)
dev_loader = make_dataloader(
    dev_samples, tokenizer, concept_to_idx, num_concepts,
    batch_size=batch_size, shuffle=False, max_length=max_length
)
val_loader = make_dataloader(
    val_samples, tokenizer, concept_to_idx, num_concepts,
    batch_size=batch_size, shuffle=False, max_length=max_length
)

concept_texts, groups, group_names = concepts_to_texts_and_groups(concepts)
task, num_outputs = infer_task_and_outputs(train_samples)
outcome_loss_fn = get_loss_fn(task)

# Initial concept anchors in the same encoder space.
concept_emb = build_concept_embeddings(
    concept_texts,
    tokenizer,
    encoder,
    DEVICE,
    batch_size=4,
    max_length=256,
    pooling="cls",
).to(DEVICE)

head = MentionAlignedAVOHead(
    concept_emb=concept_emb,
    dv=dv,
    num_outputs=num_outputs,
    temperature=attention_temperature,
    pool_temperature=pool_temperature,
    concept_bias_init=concept_bias_init,
    freeze_concepts=freeze_concepts,
).to(DEVICE)

# Simultaneous warm-up:
#   - the original [CLS] outcome head warms the shared encoder;
#   - the note-level concept loss warms Wq, Wk, and concept_bias;
#   - both losses are computed from the same note-encoder forward pass.
blackbox = BlackBoxLM(encoder, num_outputs=num_outputs).to(DEVICE)
warmup_optimizer = torch.optim.AdamW(
    [
        *blackbox.parameters(),
        *head.Wq.parameters(),
        *head.Wk.parameters(),
        head.concept_bias,
    ],
    lr=lr,
)

special_token_ids = getattr(tokenizer, "all_special_ids", [])

for epoch in range(1, joint_warmup_epochs + 1):
    blackbox.train()
    head.train()
    totals = {"loss": 0.0, "outcome": 0.0, "concept": 0.0}

    for batch in train_loader:
        tensors = {
            key: value.to(DEVICE)
            for key, value in batch.items()
            if isinstance(value, torch.Tensor)
        }

        encoder_kwargs = {
            "input_ids": tensors["input_ids"],
            "attention_mask": tensors["attention_mask"],
        }
        if tensors.get("token_type_ids") is not None:
            encoder_kwargs["token_type_ids"] = tensors["token_type_ids"]

        token_embs = blackbox.encoder(**encoder_kwargs).last_hidden_state
        outcome_logits = blackbox.head(token_embs[:, 0])

        token_mask = tensors["attention_mask"].bool()
        for token_id in special_token_ids:
            token_mask &= tensors["input_ids"] != token_id
        concept_out = head(token_embs=token_embs, token_mask=token_mask)

        labels = tensors["labels"]
        if task == "multiclass":
            outcome_loss = outcome_loss_fn(outcome_logits, labels.long())
        elif task == "multilabel":
            outcome_loss = outcome_loss_fn(outcome_logits, labels.float())
        else:
            outcome_loss = outcome_loss_fn(outcome_logits.squeeze(-1), labels.float())

        concept_loss = sampled_balanced_concept_bce_with_logits(
            concept_out.concept_logits,
            tensors["concept_labels"],
            n_hard_negatives=n_hard_concept_negatives,
            n_random_negatives=n_random_concept_negatives,
        )

        loss = outcome_loss + lambda_concept * concept_loss

        warmup_optimizer.zero_grad(set_to_none=True)
        loss.backward()
        warmup_optimizer.step()

        totals["loss"] += loss.item()
        totals["outcome"] += outcome_loss.item()
        totals["concept"] += concept_loss.item()

    # Refresh concept anchors once using the encoder from this epoch, then
    # evaluate note-level concept grounding on the development set.
    concept_emb = build_concept_embeddings(
        concept_texts,
        tokenizer,
        blackbox.encoder,
        DEVICE,
        batch_size=4,
        max_length=256,
        pooling="cls",
    ).to(DEVICE)
    with torch.no_grad():
        head.concept_emb.copy_(concept_emb)

    dev_concept_f1 = evaluate_warmup_concept_f1(
        blackbox.encoder,
        head,
        dev_loader,
        special_token_ids,
        DEVICE,
        threshold=0.5,
    )

    n_batches = max(len(train_loader), 1)
    print(
        f"Joint warm-up epoch {epoch}/{joint_warmup_epochs}: "
        f"total={totals['loss']/n_batches:.4f}, "
        f"outcome={totals['outcome']/n_batches:.4f}, "
        f"concept={totals['concept']/n_batches:.4f}, "
        f"dev_concept_micro_F1@0.5={dev_concept_f1:.4f}"
    )

# Freeze the jointly warmed encoder for the original concept-bottleneck stage.
encoder = blackbox.encoder


In [ ]:
# ============================================================
# Grid search: preserve grounding during outcome training
# ============================================================
def clone_head_state_cpu(head: nn.Module) -> Dict[str, torch.Tensor]:
    """Copy head parameters/buffers except the fixed concept anchors."""
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in head.state_dict().items()
        if name != "concept_emb"
    }


def load_partial_head_state(
    head: nn.Module,
    state: Dict[str, torch.Tensor],
) -> None:
    missing, unexpected = head.load_state_dict(state, strict=False)
    if unexpected or set(missing) - {"concept_emb"}:
        raise RuntimeError(
            f"Head-state mismatch; missing={missing}, unexpected={unexpected}"
        )


def build_model_from_warmup() -> MentionAlignedAVOModel:
    """Rebuild an identical model from the shared jointly warmed checkpoint."""
    new_head = MentionAlignedAVOHead(
        concept_emb=head.concept_emb.detach(),
        dv=dv,
        num_outputs=num_outputs,
        temperature=attention_temperature,
        pool_temperature=pool_temperature,
        concept_bias_init=head.concept_bias.detach(),
        freeze_concepts=freeze_concepts,
    ).to(DEVICE)
    load_partial_head_state(new_head, warmed_head_state)

    return MentionAlignedAVOModel(
        text_encoder=encoder,
        head=new_head,
        special_token_ids=getattr(tokenizer, "all_special_ids", []),
        freeze_text_encoder=freeze_text_encoder,
    ).to(DEVICE)


def grounding_and_outcome_parameters(
    model: MentionAlignedAVOModel,
) -> Tuple[List[nn.Parameter], List[nn.Parameter]]:
    grounding_parameters = (
        list(model.head.Wq.parameters())
        + list(model.head.Wk.parameters())
        + [model.head.concept_bias]
    )
    if isinstance(model.head.concept_emb, nn.Parameter):
        grounding_parameters.append(model.head.concept_emb)

    outcome_parameters = (
        list(model.head.Wv.parameters())
        + [model.head.O]
        + ([] if model.head.bias is None else [model.head.bias])
    )
    return grounding_parameters, outcome_parameters


def set_trainable(parameters: List[nn.Parameter], trainable: bool) -> None:
    for parameter in parameters:
        parameter.requires_grad_(trainable)


# Save the common starting point after simultaneous outcome/concept warm-up.
warmed_head_state = clone_head_state_cpu(head)
group_row_indices = prepare_group_index_tensors(groups, DEVICE)

grid_rows: List[Dict[str, Any]] = []
grid_checkpoints: Dict[Tuple[str, int], Dict[str, torch.Tensor]] = {}

for config_index, config in enumerate(grounding_grid):
    print(f"\n===== {config['name']} =====")
    set_global_seed(SEED, deterministic=True)

    candidate_model = build_model_from_warmup()
    grounding_parameters, outcome_parameters = grounding_and_outcome_parameters(
        candidate_model
    )

    # Keep both groups in the optimizer so freeze-then-unfreeze does not
    # require rebuilding the optimizer.
    optimizer = torch.optim.AdamW(
        [
            {
                "params": grounding_parameters,
                "lr": config["grounding_lr"],
            },
            {
                "params": outcome_parameters,
                "lr": config["outcome_lr"],
            },
        ]
    )

    for epoch in range(1, num_epochs + 1):
        grounding_trainable = (
            epoch > config["freeze_grounding_epochs"]
            and config["grounding_lr"] > 0
        )
        set_trainable(grounding_parameters, grounding_trainable)
        optimizer.param_groups[0]["lr"] = (
            config["grounding_lr"] if grounding_trainable else 0.0
        )

        candidate_model.train()
        totals = {
            "loss": 0.0,
            "outcome": 0.0,
            "concept": 0.0,
            "group": 0.0,
        }

        for batch in train_loader:
            tensors = {
                key: value.to(DEVICE)
                for key, value in batch.items()
                if isinstance(value, torch.Tensor)
            }
            out, _ = candidate_model(
                input_ids=tensors["input_ids"],
                attention_mask=tensors["attention_mask"],
                token_type_ids=tensors.get("token_type_ids"),
            )

            labels = tensors["labels"]
            if task == "multiclass":
                outcome_loss = outcome_loss_fn(out.logits, labels.long())
            elif task == "multilabel":
                outcome_loss = outcome_loss_fn(out.logits, labels.float())
            else:
                outcome_loss = outcome_loss_fn(
                    out.logits.squeeze(-1), labels.float()
                )

            if grounding_trainable and config["lambda_concept"] > 0:
                concept_loss = sampled_balanced_concept_bce_with_logits(
                    out.concept_logits,
                    tensors["concept_labels"],
                    n_hard_negatives=n_hard_concept_negatives,
                    n_random_negatives=n_random_concept_negatives,
                )
            else:
                concept_loss = out.logits.new_zeros(())

            beta = out.V @ out.O
            group_loss = group_lasso_penalty_on_beta(
                beta, group_row_indices, group_lasso_cfg
            )

            loss = (
                outcome_loss
                + config["lambda_concept"] * concept_loss
                + group_loss
            )

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            totals["loss"] += loss.item()
            totals["outcome"] += outcome_loss.item()
            totals["concept"] += concept_loss.item()
            totals["group"] += group_loss.item()

        dev_metrics = evaluate_val(
            candidate_model, dev_loader, task, DEVICE
        )
        n_batches = max(len(train_loader), 1)

        row = {
            "config": config["name"],
            "epoch": epoch,
            "grounding_trainable": grounding_trainable,
            "grounding_lr": optimizer.param_groups[0]["lr"],
            "outcome_lr": config["outcome_lr"],
            "lambda_concept": config["lambda_concept"],
            "train_loss": totals["loss"] / n_batches,
            "train_outcome_loss": totals["outcome"] / n_batches,
            "train_concept_loss": totals["concept"] / n_batches,
            "train_group_loss": totals["group"] / n_batches,
            **dev_metrics,
        }
        grid_rows.append(row)
        grid_checkpoints[(config["name"], epoch)] = clone_head_state_cpu(
            candidate_model.head
        )

        print(
            f"epoch={epoch}/{num_epochs}, "
            f"grounding_trainable={grounding_trainable}, "
            f"AUROC={dev_metrics.get('AUROC', float('nan')):.4f}, "
            f"concept_F1={dev_metrics['concept_micro_F1@0.5']:.4f}, "
            f"presence_hit@1={dev_metrics['concept_presence_hit@1']:.4f}"
        )

    del candidate_model, optimizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


grid_results = pd.DataFrame(grid_rows)

summary_columns = [
    "config",
    "epoch",
    "grounding_trainable",
    "grounding_lr",
    "outcome_lr",
    "lambda_concept",
    "AUROC",
    "AUPR",
    "concept_micro_F1@0.5",
    "concept_presence_hit@1",
]
print("\n===== Development grid results =====")
display(
    grid_results[summary_columns]
    .sort_values(
        ["AUROC", "concept_micro_F1@0.5", "concept_presence_hit@1"],
        ascending=False,
    )
    .reset_index(drop=True)
)

# Preserve predictive performance first, then maximize grounding.
valid_results = grid_results.dropna(subset=["AUROC"]).copy()
if valid_results.empty:
    raise RuntimeError("No finite development AUROC was produced.")

best_dev_auroc = valid_results["AUROC"].max()
eligible_results = valid_results[
    valid_results["AUROC"]
    >= best_dev_auroc - outcome_auroc_tolerance
]

best_row = (
    eligible_results
    .sort_values(
        [
            "concept_micro_F1@0.5",
            "concept_presence_hit@1",
            "AUROC",
        ],
        ascending=False,
    )
    .iloc[0]
)

best_key = (str(best_row["config"]), int(best_row["epoch"]))
print(
    "\nSelected checkpoint:",
    {
        "config": best_key[0],
        "epoch": best_key[1],
        "dev_AUROC": round(float(best_row["AUROC"]), 4),
        "dev_concept_F1": round(
            float(best_row["concept_micro_F1@0.5"]), 4
        ),
        "dev_presence_hit@1": round(
            float(best_row["concept_presence_hit@1"]), 4
        ),
    },
)

# Restore the selected development checkpoint and evaluate the test set once.
model = build_model_from_warmup()
load_partial_head_state(model.head, grid_checkpoints[best_key])

val_metrics = evaluate_val(model, val_loader, task, DEVICE)
print(
    "Final validation/test:",
    {key: round(value, 4) for key, value in val_metrics.items()},
)
